# 04 — Building-weighted noise exposure (Ohio counties)

**Question:** What share of each Ohio county's residents live in areas with high transportation noise, when population is placed at building locations instead of block centers?

**Why this notebook exists:** Notebook 03 assigned each census block's population to its internal point. In large rural blocks that point falls mid-block, away from the roadside homes where people actually live, so exposure was underestimated in exurban counties (e.g., Geauga: 0.1% of residents vs. 6.3% of land at ≥45 dBA). This notebook places population at buildings to correct that. Notebook 03's results are kept as a sensitivity analysis.

**Data:**
- **Noise:** BTS National Transportation Noise Map, 2022/2023, road, rail, and aviation combined (Ohio layer). 24-hour LAeq (dBA), 30 m grid; levels below 45 dBA are not mapped.
- **Population:** 2020 Census blocks for Ohio (TIGER/Line 2022 release), `POP20`.
- **Buildings:** Microsoft US Building Footprints, Ohio (5,544,032 footprints; imagery mostly 2019–2020, some older).

**Method:**
1. Convert each building footprint to a point at its center, in the noise raster's coordinate system.
2. Drop very small footprints (under 35 m²), which are mostly sheds and garages.
3. Assign each building to its census block with a spatial join.
4. Split each block's population evenly among its buildings. Populated blocks with no buildings fall back to the block's internal point.
5. Look up noise at each building and compute, for each county, the share of residents at or above 45, 50, and 55 dBA.
6. Compare with the internal-point results from notebook 03.

**Output:** `data/processed/noise_oh_county_2022_bldg.csv`: one row per county with population and the share exposed at each threshold (building-weighted method).

**Caveats:** Footprints include non-residential buildings, and population is split evenly regardless of building size or use.

In [1]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

# --- Paths (copy BLOCKS_PATH exactly as it is in notebook 03) ---
NOISE_DIR     = Path("../data/large/bts_2022_combined")
root          = NOISE_DIR / "CONUS_rail_road_and_aviation_noise_2022"
GDB           = root / "CONUS_rail_road_aviation_noise.Overviews" / "CONUS_rail_road_aviation_noise.gdb"
BLOCKS_PATH   = Path("../data/large/census_blocks/tl_2022_39_tabblock20")
BUILD_GEOJSON = Path("../data/large/buildings/Ohio.geojson")
BUILD_POINTS  = Path("../data/large/buildings/ohio_building_points.parquet")
RAW           = Path("../data/raw")
PROCESSED     = Path("../data/processed")

# Fail early, with a clear message, if any input is missing
for name, p in [("GDB", GDB), ("BLOCKS_PATH", BLOCKS_PATH), ("BUILD_GEOJSON", BUILD_GEOJSON)]:
    assert p.exists(), f"{name} not found: {p}"

# --- Ohio noise raster and its coordinate system ---
with rasterio.open(GDB) as gdb_src:
    OH_PATH = next(sd for sd in gdb_src.subdatasets if sd.endswith(":OH_rail_road_aviation_noise"))

with rasterio.open(OH_PATH) as src:
    RASTER_CRS = src.crs.to_wkt()

c:\Users\888am\projects\noise-cvd\.venv\Lib\site-packages\rasterio\__init__.py:379: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


In [3]:
import time
import geopandas as gpd
import rasterio
from pathlib import Path

BUILD_GEOJSON = Path("../data/large/buildings/Ohio.geojson")
BUILD_POINTS  = Path("../data/large/buildings/ohio_building_points.parquet")

if BUILD_POINTS.exists():
    # Fast path: reuse the points saved on a previous run
    t0 = time.time()
    bpts = gpd.read_parquet(BUILD_POINTS)
    print(f"Loaded {len(bpts):,} building points from Parquet in {time.time() - t0:.0f} s")
else:
    # Slow path (first run only): read footprints, convert to points, save
    t0 = time.time()
    bld = gpd.read_file(BUILD_GEOJSON)
    print(f"Loaded {len(bld):,} buildings from GeoJSON in {time.time() - t0:.0f} s")

    bld = bld.to_crs(RASTER_CRS)
    bld["area_m2"] = bld.geometry.area

    bpts = gpd.GeoDataFrame(
        {"area_m2": bld["area_m2"].values},
        geometry=bld.geometry.centroid.values,
        crs=RASTER_CRS,
    )
    del bld

    bpts.to_parquet(BUILD_POINTS)
    print(f"Saved building points to {BUILD_POINTS}")

# Checks that apply either way
assert len(bpts) == 5_544_032, len(bpts)
print(bpts["area_m2"].describe().round(1))

Loaded 5,544,032 building points from Parquet in 3 s
count    5544032.0
mean         235.0
std          951.3
min            5.0
25%          101.1
50%          154.3
75%          230.5
max       256146.8
Name: area_m2, dtype: float64


In [4]:
for a in [20, 35, 50]:
    print(f"< {a} m²: {(bpts['area_m2'] < a).mean():.1%} of buildings")

MIN_AREA = 35   # update if the numbers above suggest otherwise
bpts = bpts[bpts["area_m2"] >= MIN_AREA].copy()
print(f"{len(bpts):,} buildings kept")

< 20 m²: 0.2% of buildings
< 35 m²: 1.8% of buildings
< 50 m²: 5.5% of buildings
5,444,424 buildings kept


In [5]:
# Block outlines this time (not just internal points), in the raster's CRS
blocks = gpd.read_file(
    BLOCKS_PATH, columns=["GEOID20", "POP20", "INTPTLAT20", "INTPTLON20"]
).to_crs(RASTER_CRS)
assert blocks["POP20"].sum() == 11_799_448

t0 = time.time()
bj = gpd.sjoin(bpts, blocks[["GEOID20", "geometry"]], how="left", predicate="within")
print(f"Spatial join took {time.time() - t0:.0f} s")

assert bj.index.is_unique, "some buildings matched more than one block"
print(f"Buildings not inside any block: {bj['GEOID20'].isna().mean():.2%}")

Spatial join took 11 s
Buildings not inside any block: 0.32%


In [6]:
bj = bj.dropna(subset=["GEOID20"])

# How many buildings each block has
counts = bj.groupby("GEOID20").size().rename("n_bldg")
blocks = blocks.merge(counts, left_on="GEOID20", right_index=True, how="left")
blocks["n_bldg"] = blocks["n_bldg"].fillna(0).astype(int)

# Populated blocks with no buildings -> fall back to the internal point
pop_blocks = blocks[blocks["POP20"] > 0]
no_bldg = pop_blocks[pop_blocks["n_bldg"] == 0]
print(f"Populated blocks with no buildings: {len(no_bldg):,} "
      f"({no_bldg['POP20'].sum() / 11_799_448:.2%} of Ohio's population)")

# Each building carries an equal share of its block's population
bj = bj.merge(blocks[["GEOID20", "POP20", "n_bldg"]], on="GEOID20")
bj["pop"] = bj["POP20"] / bj["n_bldg"]
bj = bj[bj["pop"] > 0]          # buildings in zero-population blocks carry nobody

fallback = gpd.GeoDataFrame(
    {"GEOID20": no_bldg["GEOID20"].values, "pop": no_bldg["POP20"].values},
    geometry=gpd.points_from_xy(no_bldg["INTPTLON20"].astype(float), no_bldg["INTPTLAT20"].astype(float)),
    crs="EPSG:4269",
).to_crs(RASTER_CRS)

# One table of "where people are": building points plus fallback points
people = gpd.GeoDataFrame(
    pd.concat([bj[["GEOID20", "pop", "geometry"]], fallback], ignore_index=True),
    geometry="geometry", crs=RASTER_CRS,
)
people["county_fips"] = people["GEOID20"].str[:5]

# Everyone is still counted exactly once
assert np.isclose(people["pop"].sum(), 11_799_448), people["pop"].sum()
print(f"{len(people):,} population points")

Populated blocks with no buildings: 2,403 (0.48% of Ohio's population)
5,349,093 population points


In [7]:
coords = np.column_stack([people.geometry.x, people.geometry.y])

t0 = time.time()
with rasterio.open(OH_PATH) as src:
    vals = np.fromiter((v[0] for v in src.sample(coords)), dtype="float64", count=len(coords))
print(f"Sampled {len(vals):,} points in {time.time() - t0:.0f} s")

vals[vals > 1e30] = np.nan                  # NoData -> below 45 dBA
assert np.nanmax(vals) < 120, np.nanmax(vals)
people["laeq"] = vals

share_mapped = people.loc[people["laeq"].notna(), "pop"].sum() / people["pop"].sum()
print(f"Share of Ohio residents at >=45 dBA: {share_mapped:.3f}")

Sampled 5,349,093 points in 90 s
Share of Ohio residents at >=45 dBA: 0.236


In [8]:
THRESHOLDS = [45, 50, 55]

for t in THRESHOLDS:
    people[f"pop_ge{t}"] = people["pop"] * (people["laeq"] >= t)

cols = ["pop"] + [f"pop_ge{t}" for t in THRESHOLDS]
county_b = (people.groupby("county_fips")[cols].sum()
                  .rename(columns={"pop": "POP20"})
                  .reset_index())

for t in THRESHOLDS:
    county_b[f"pct_ge{t}"] = county_b[f"pop_ge{t}"] / county_b["POP20"]

In [9]:
pct_cols = [f"pct_ge{t}" for t in THRESHOLDS]

assert len(county_b) == 88, len(county_b)
assert np.isclose(county_b["POP20"].sum(), 11_799_448), county_b["POP20"].sum()
assert county_b[pct_cols].ge(0).all().all() and county_b[pct_cols].le(1).all().all()
assert (county_b["pct_ge45"] >= county_b["pct_ge50"]).all()
assert (county_b["pct_ge50"] >= county_b["pct_ge55"]).all()

county_b[pct_cols].describe().round(3)

,pct_ge45,pct_ge50,pct_ge55
count,88.000,88.000,88.000
mean,0.193,0.081,0.025
std,0.125,0.054,0.015
min,0.041,0.021,0.003
25%,0.091,0.045,0.014
50%,0.154,0.057,0.020
75%,0.256,0.108,0.032
max,0.546,0.242,0.077


In [10]:
names = (pd.read_csv(RAW / "places_2025_county_OH.csv", dtype={"locationid": str})
           [["locationid", "locationname"]].drop_duplicates()
           .rename(columns={"locationid": "county_fips", "locationname": "county"}))

county_b = county_b.merge(names, on="county_fips", how="left")
assert county_b["county"].notna().all()

county_b.to_csv(PROCESSED / "noise_oh_county_2022_bldg.csv", index=False)

In [11]:
old = pd.read_csv(PROCESSED / "noise_oh_county_2022.csv", dtype={"county_fips": str})

cmp_df = county_b.merge(
    old[["county_fips"] + pct_cols], on="county_fips", suffixes=("_bldg", "_pt")
)
assert len(cmp_df) == 88

# Agreement between methods: Pearson on values, Spearman on rankings
for t in THRESHOLDS:
    a, b = cmp_df[f"pct_ge{t}_pt"], cmp_df[f"pct_ge{t}_bldg"]
    print(f">= {t} dBA: Pearson r = {a.corr(b):.2f}, Spearman rho = {a.rank().corr(b.rank()):.2f}, "
          f"median county change = {(b - a).median():+.3f}")

check_names = ["Cuyahoga", "Franklin", "Hamilton", "Geauga", "Medina", "Seneca", "Crawford"]
(cmp_df.set_index("county")
       .loc[check_names, ["pct_ge45_pt", "pct_ge45_bldg", "pct_ge55_pt", "pct_ge55_bldg"]]
       .round(3))

>= 45 dBA: Pearson r = 1.00, Spearman rho = 0.98, median county change = +0.051
>= 50 dBA: Pearson r = 0.99, Spearman rho = 0.96, median county change = +0.032
>= 55 dBA: Pearson r = 0.95, Spearman rho = 0.85, median county change = +0.011


,pct_ge45_pt,pct_ge45_bldg,pct_ge55_pt,pct_ge55_bldg
county,,,,
Cuyahoga,0.352,0.386,0.025,0.038
Franklin,0.288,0.321,0.022,0.032
Hamilton,0.211,0.267,0.015,0.027
Geauga,0.001,0.047,0.000,0.010
Medina,0.019,0.068,0.001,0.008
Seneca,0.411,0.467,0.059,0.077
Crawford,0.447,0.493,0.050,0.063


In [12]:
people[["county_fips", "pop", "geometry"]].to_parquet(Path("../data/large/buildings/ohio_people_points.parquet"))

## Interpretation

### Findings
- Population was placed at **5,349,093 points**: building centers (footprints ≥35 m², dropping 1.8% of buildings) plus internal points for the 2,403 populated blocks with no buildings (0.48% of Ohio's population).
- Statewide, **23.6% of Ohio residents** live at ≥45 dBA.
- Median county share exposed: **15.4%** at ≥45 dBA, **5.7%** at ≥50, **2.0%** at ≥55. The least-exposed county is now at 4.1% (≥45 dBA), compared with 0.1% under the internal-point method.

### Comparison with the internal-point method (notebook 03)
| Threshold | Pearson r | Spearman ρ | Median county change |
|---|---|---|---|
| ≥45 dBA | 1.00 | 0.98 | +5.1 pts |
| ≥50 dBA | 0.99 | 0.96 | +3.2 pts |
| ≥55 dBA | 0.95 | 0.85 | +1.1 pts |

- Exposure rose in every county, and most in exurban counties (Geauga 0.1% → 4.7%; Medina 1.9% → 6.8%), confirming that internal points missed roadside homes.
- County **rankings are nearly unchanged at 45 and 50 dBA**, so results at those thresholds should be robust to the allocation method. Agreement is weaker at 55 dBA, where shares are small.
- Geauga remains among the least exposed even with buildings, so its low exposure reflects the BTS road model (thin noise bands on rural roads), not population placement.
- Rail-junction counties (Crawford, Seneca) stay most exposed under both methods; the rail explanation is to be confirmed with the rail-only layer.

### Caveats
- Footprints include non-residential buildings, and each block's population is split evenly among its buildings regardless of size or use.
- 0.32% of buildings fell outside any block and were dropped; no residents were lost, since population comes from blocks.

### Decision
Building-weighted exposure is the **primary** exposure measure; internal-point exposure (notebook 03) is the sensitivity analysis.

### Output
`data/processed/noise_oh_county_2022_bldg.csv`